In [1]:
import yfinance as yf
import pandas as pd

stocks = {
    "Nvidia" : "NVDA",
    "Apple" : "AAPL",
    "Microsoft" : "MSFT",
    "AMD" : "AMD",
    "Intel" : "INTC",
    "Cisco" : "CSCO",
    "Oracle" : "ORCL",
    "IBM" : "IBM",
    "Adobe" : "ADBE",
    "Salesforce" : "CRM",
    "Uber" : "UBER",
    "JP Morgan" : "JPM",
    "Visa" : "V",
    "Mastercard" : "MA",
    "Wells Fargo" : "WFC",
    "Morgan Stanley" : "MS",
    "American Express" : "AXP",
    "Bank of America" : "BAC",
    "Amazon" : "AMZN",
    "Tesla" : "TSLA",
    "Meta" : "META",
    "Google" : "GOOGL",
    "Netflix" : "NFLX",
    "Disney" : "DIS",
    "Verizon" : "VZ",
    "Costco" : "COST",
    "Sandisk" : "SNDK",
    "HP" : "HPE",
    "Motorola" : "MSI",
    "Airbnb" : "ABNB",
    "DoorDash" : "DASH"
}

actions = {
    "Overweight": "buy",
    "Outperform": "buy",
    "Buy": "buy",
    "Strong Buy": "buy",
    "Positive": "buy",
    "Market Outperform": "buy",
    "Sector Outperform": "buy",
    "Accumulate": "buy",
    "Outperformer": "buy",
    "Top Pick": "buy",
    "Long-Term Buy": "buy",
    "Neutral": "hold",
    "Hold": "hold",
    "Market Perform": "hold",
    "Equal-Weight": "hold",
    "Equal-weight": "hold",
    "Perform": "hold",
    "Peer Perform": "hold",
    "Sector Weight": "hold",
    "Sector Perform": "hold",
    "Fair Value": "hold",
    "Average": "hold",
    "In-Line": "hold",
    "Market Weight": "hold",
    "Mixed": "hold",
    "Underperform": "sell",
    "Underweight": "sell",
    "Reduce": "sell",
    "Sell": "sell",
    "Market Underperform": "sell",
    "Negative": "sell",
}

def get_forward_return(ticker, event_date, months):
    df = pricedata[ticker]

    df = df.copy()
    df.index = df.index.tz_localize(None)
    event_date = pd.to_datetime(event_date).tz_localize(None)
    target_date = event_date + pd.DateOffset(months = months)
    start_slice = df[df.index >= event_date]
    end_slice = df[df.index >= target_date]
    if start_slice.empty or end_slice.empty:
        return None
    start_price = start_slice.iloc[0]['Close']
    end_price = end_slice.iloc[0]['Close']

    return (end_price - start_price) / start_price


def get_spy_return(event_date, months):
    df = spy_df.copy()
    df.index = df.index.tz_localize(None)
    event_date = pd.to_datetime(event_date).tz_localize(None)
    target_date = event_date + pd.DateOffset(months = months)
    start_slice = df[df.index >= event_date]
    end_slice = df[df.index <= event_date]
    if start_slice.empty or end_slice.empty:
        return None
    start_price = start_slice.iloc[0]['Close']
    end_price = end_slice.iloc[0]['Close']
    return (end_price - start_price) / start_price

spy_df = yf.Ticker("^GSPC").history(period="13y")

pricedata = {}
for i in stocks.values():
    pricedata[i] = yf.Ticker(i).history(period="13y")

all = []
for i in stocks.values():
    df = yf.Ticker(i).upgrades_downgrades
    df = df.reset_index()
    df = df[df['ToGrade'] != '']
    df['rating'] = df['ToGrade'].map(actions)
    df = df.dropna(subset=['rating'])
    df['ticker'] = i
    all.append(df)

master = pd.concat(all, ignore_index=True)
master['ret_1m'] = master.apply(lambda row: get_forward_return(row['ticker'], row['GradeDate'], 1), axis=1)
master['ret_3m'] = master.apply(lambda row: get_forward_return(row['ticker'], row['GradeDate'], 3), axis=1)
master['ret_6m'] = master.apply(lambda row: get_forward_return(row['ticker'], row['GradeDate'], 6), axis=1)
master['spy_ret_1m'] = master['GradeDate'].apply(lambda d: get_spy_return(d, 1))
master['spy_ret_3m'] = master['GradeDate'].apply(lambda d: get_spy_return(d, 3))
master['spy_ret_6m'] = master['GradeDate'].apply(lambda d: get_spy_return(d, 6))
master['excess_ret_1m'] = master['ret_1m'] - master['spy_ret_1m']
master['excess_ret_3m'] = master['ret_3m'] - master['spy_ret_3m']
master['excess_ret_6m'] = master['ret_6m'] - master['spy_ret_6m']
master.head()

/Users/pranavmandala/Development/projects/analyst-rating-accuracy/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


,GradeDate,Firm,ToGrade,FromGrade,Action,priceTargetAction,currentPriceTarget,priorPriceTarget,rating,ticker,ret_1m,ret_3m,ret_6m,spy_ret_1m,spy_ret_3m,spy_ret_6m,excess_ret_1m,excess_ret_3m,excess_ret_6m
0,2026-09-10 14:02:40,Piper Sandler,Overweight,,init,Announces,300.0,0.0,buy,NVDA,NaN,NaN,NaN,-0.778294,-0.778294,-0.778294,NaN,NaN,NaN
1,2026-09-04 11:38:09,Rosenblatt,Buy,Buy,main,Maintains,390.0,390.0,buy,NVDA,NaN,NaN,NaN,-0.778772,-0.778772,-0.778772,NaN,NaN,NaN
2,2026-09-04 11:04:59,Needham,Buy,Buy,reit,Maintains,300.0,300.0,buy,NVDA,NaN,NaN,NaN,-0.778772,-0.778772,-0.778772,NaN,NaN,NaN
3,2026-08-27 19:31:05,Citigroup,Buy,Buy,main,Raises,315.0,300.0,buy,NVDA,NaN,NaN,NaN,-0.779869,-0.779869,-0.779869,NaN,NaN,NaN
4,2026-08-27 18:33:14,Mizuho,Outperform,Outperform,main,Raises,315.0,300.0,buy,NVDA,NaN,NaN,NaN,-0.779869,-0.779869,-0.779869,NaN,NaN,NaN


In [2]:
from scipy import stats

In [9]:
buy_returns = master[master["rating"] == "buy"]["excess_ret_1m"].dropna()
sell_returns = master[master["rating"] == "sell"]["excess_ret_1m"].dropna()
t_stat, p_value = stats.ttest_ind(buy_returns, sell_returns, equal_var = False)
print(f"t-statistic: {t_stat:.3f}")
print(f"p-value: {p_value:.4f}")

t-statistic: 4.140
p-value: 0.0000


In [10]:
buy_returns = master[master["rating"] == "buy"]["excess_ret_3m"].dropna()
sell_returns = master[master["rating"] == "sell"]["excess_ret_3m"].dropna()
t_stat, p_value = stats.ttest_ind(buy_returns, sell_returns, equal_var = False)
print(f"t-statistic: {t_stat:.3f}")
print(f"p-value: {p_value:.4f}")

t-statistic: 1.808
p-value: 0.0710


In [11]:
buy_returns = master[master["rating"] == "buy"]["excess_ret_6m"].dropna()
sell_returns = master[master["rating"] == "sell"]["excess_ret_6m"].dropna()
t_stat, p_value = stats.ttest_ind(buy_returns, sell_returns, equal_var = False)
print(f"t-statistic: {t_stat:.3f}")
print(f"p-value: {p_value:.4f}")

t-statistic: -1.689
p-value: 0.0916


In [12]:
buy_returns = master[master["rating"] == "buy"]["excess_ret_1m"].dropna()
sell_returns = master[master["rating"] == "hold"]["excess_ret_1m"].dropna()
t_stat, p_value = stats.ttest_ind(buy_returns, sell_returns, equal_var = False)
print(f"t-statistic: {t_stat:.3f}")
print(f"p-value: {p_value:.4f}")

t-statistic: 8.074
p-value: 0.0000


In [13]:
buy_returns = master[master["rating"] == "buy"]["excess_ret_3m"].dropna()
sell_returns = master[master["rating"] == "hold"]["excess_ret_3m"].dropna()
t_stat, p_value = stats.ttest_ind(buy_returns, sell_returns, equal_var = False)
print(f"t-statistic: {t_stat:.3f}")
print(f"p-value: {p_value:.4f}")

t-statistic: 5.607
p-value: 0.0000


In [14]:
buy_returns = master[master["rating"] == "buy"]["excess_ret_6m"].dropna()
sell_returns = master[master["rating"] == "hold"]["excess_ret_6m"].dropna()
t_stat, p_value = stats.ttest_ind(buy_returns, sell_returns, equal_var = False)
print(f"t-statistic: {t_stat:.3f}")
print(f"p-value: {p_value:.4f}")

t-statistic: 2.287
p-value: 0.0222


In [15]:
buy_returns = master[master["rating"] == "sell"]["excess_ret_1m"].dropna()
sell_returns = master[master["rating"] == "hold"]["excess_ret_1m"].dropna()
t_stat, p_value = stats.ttest_ind(buy_returns, sell_returns, equal_var = False)
print(f"t-statistic: {t_stat:.3f}")
print(f"p-value: {p_value:.4f}")

t-statistic: -0.690
p-value: 0.4901
